In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from sentence_transformers import SentenceTransformer

from src.data import load_dataset
from src.embedder import record_to_text, get_embeddings, save_embeddings, load_embeddings, get_cache_path
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import MODELS, TOP_K

corpus, questions, categories = load_dataset()
corpus_texts = [record_to_text(item) for item in corpus]

print(f"Корпус: {len(corpus)} фрагментов")
print(f"Вопросы: {len(questions)} вопросов")

Корпус: 200 фрагментов
Вопросы: 25 вопросов


In [2]:
model_name = MODELS[1]
cache_path = get_cache_path(model_name)

if Path(cache_path).exists():
    corpus_embedding = load_embeddings(cache_path)
    print("Загруженна из кэша")
else:
    corpus_embedding = get_embeddings(corpus_texts, model_name)
    save_embeddings(cache_path, corpus_embedding)
    print("Вычислено и добавлено в кэш")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

D:\Polytech\2_sem\Rostelecom\semantic-search-case\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dakar\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Вычислено и добавлено в кэш


In [3]:
model = SentenceTransformer(model_name)
search_outputs = search_all_questions(questions, model, corpus, corpus_embedding)

metrics = evaluate(search_outputs, k=TOP_K)
print(metrics)
print("Для модели:", model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'precision_at_3': 0.92, 'mrr': 0.6333333333333333, 'num_questions': 25}
Для модели: paraphrase-multilingual-mpnet-base-v2
